In [ ]:
# ==============================
# Import Required Modules
# ==============================
import time
import math
from tabulate import tabulate
# ===================================
# Graph Creation (Dynamic or Default)
# ===================================


def create_graph():
    graph = {}


# ==============================
# Number of Locations
# ==============================
    while True:
        n_str = input("👉 Enter number of locations: ").strip()
        if n_str.isdigit():
            n = int(n_str)
            if n < 2:
                print("⚠️ You must create at least 2 locations. Please try again.\n")
                continue
            print(f"\n✅ Great! We’ll create {n} unique locations.")
            break
        else:
            print("⚠️ Please enter a valid positive integer.\n")

    # ==============================
    # Number of Edges
    # ==============================
    print("\n🗺️ Let's build your map of locations!")
    graph = {}

    for i in range(1, n + 1):
        while True:
            print(f"\n➡️ Step {i}/{n}")
            if graph:
                print("   Locations added so far:", ", ".join(graph.keys()))
            node = input(f"👉 Enter name for location {i}: ").strip()

            if not node:
                print("⚠️ Location name cannot be empty. Please try again.")
                continue
            if node in graph:
                print("⚠️ This location already exists. Please enter a unique name.")
                continue

            graph[node] = {}
            print(f"✅ Added '{node}' successfully!")
            break

    print("\n🎉 All locations created successfully!")
    print("   Final set of locations:", ", ".join(graph.keys()))

    # ==============================
    # Edges (Roads) Input
    # ==============================

    if len(graph) < 2:
        print("⚠️ You need at least 2 locations to add roads.")
    else:
        while True:
            e_str = input(
                "\n📌 How many roads (edges) do you want to add? ").strip()
            if e_str.isdigit() and int(e_str) > 0:
                e = int(e_str)
                max_possible = len(graph) * (len(graph) -
                                             1)  # directed edges max
                if e > max_possible:
                    print(
                        f"⚠️ Too many roads. With {len(graph)} locations, you can add at most {max_possible} roads.")
                    continue
                print(f"\n✅ Okay, we’ll add {e} road(s).")
                break
            else:
                print("⚠️ Please enter a valid positive integer (at least 1).\n")

        for j in range(1, e + 1):
            print(f"\n➡️ Road {j}/{e}")
            while True:
                src = input("   🏁 From location: ").strip()
                dst = input("   🎯 To location: ").strip()
                cost = input("   💰 Cost/Distance: ").strip()

                if src not in graph or dst not in graph:
                    print("⚠️ Both locations must exist in the graph. Try again.")
                    continue
                if src == dst:
                    print("⚠️ A road cannot connect a location to itself. Try again.")
                    continue
                if not cost.isdigit() or int(cost) <= 0:
                    print("⚠️ Cost must be a positive integer. Try again.")
                    continue

            # Prevent duplicate edge overwrite
                if dst in graph[src]:
                    print(
                        f"⚠️ Road {src} ➝ {dst} already exists with cost {graph[src][dst]}.")
                    overwrite = input(
                        "   Do you want to overwrite it? (y/n): ").strip().lower()
                    if overwrite != "y":
                        print(
                            "❌ Road not updated, please enter a different connection.")
                        continue

                graph[src][dst] = int(cost)
                print(f"✅ Added road: {src} ➝ {dst} (Cost: {cost})")
                break

        # ==============================
        # Final Graph Preview
        # ==============================
        print("\n🎉 Map creation complete! Here’s your graph:")
        for location, roads in graph.items():
            if roads:
                print(f"   {location} ➝ {roads}")
            else:
                print(f"   {location} ➝ No outgoing roads")
    return graph


# Predefined Delhi map
default_graph = {
    "Indira Gandhi Airport": {"India Gate": 15, "Qutub Minar": 20},
    "India Gate": {"Indira Gandhi Airport": 15, "Lotus Temple": 15, "Humayun's Tomb": 10, "Red Fort": 10},
    "Red Fort": {"India Gate": 10, "Jama Masjid": 5, "Lotus Temple": 30},
    "Qutub Minar": {"Indira Gandhi Airport": 20, "Lotus Temple": 15, "Humayun's Tomb": 20},
    "Lotus Temple": {"India Gate": 15, "Qutub Minar": 15, "Akshardham Temple": 25, "Humayun's Tomb": 10, "Red Fort": 30},
    "Jama Masjid": {"Red Fort": 5, "Akshardham Temple": 20},
    "Humayun's Tomb": {"India Gate": 10, "Lotus Temple": 10, "Qutub Minar": 20, "Akshardham Temple": 15},
    "Akshardham Temple": {"Jama Masjid": 20, "Lotus Temple": 25, "Humayun's Tomb": 15}
}

choice = input("Do you want to use the default graph? (y/n): ").strip().lower()
graph = default_graph if choice == 'y' else create_graph()

nodes_list = list(graph.keys())
print("\nGraph Nodes:")
for idx, node in enumerate(nodes_list, 1):
    print(f"{idx}. {node}")

# ==============================
# Start Node Choice
# ==============================
while True:
    start_idx_str = input("Enter the START node index: ").strip()
    if not start_idx_str.isdigit():
        print("⚠️ Please enter a valid number.")
        continue

    start_idx = int(start_idx_str) - 1
    if 0 <= start_idx < len(nodes_list):
        start_node = nodes_list[start_idx]
        print("\n✅ Start Node:", start_node)
        break
    else:
        print(
            f"⚠️ Invalid index. Please enter a number between 1 and {len(nodes_list)}.")

# =====================================
# Transition Model / Successor Function
# =====================================


def successors(graph, node, visited=None):
    return [(nbr, w) for nbr, w in graph[node].items() if not visited or nbr not in visited]


def print_transition_model(graph):
    table = []
    for node in graph:
        neighbors = ", ".join(
            [f"{nbr}({cost})" for nbr, cost in graph[node].items()])
        # wrap neighbors for better readability
        if len(neighbors) > 40:
            neighbors = "\n".join(
                [f"{nbr}({cost})" for nbr, cost in graph[node].items()])
        table.append([node, neighbors])
    print("\n=== Transition Model / Successor Function ===")
    print(tabulate(table, headers=[
          "Node", "Neighbors (Cost)"], tablefmt="grid"))


print_transition_model(graph)

# ==============================
# Cost / Transition Matrix
# ==============================


def get_initials(name):
    """Convert a node name to initials (first letter of each word)."""
    return "".join(word[0].upper() for word in name.split())


def print_cost_matrix(graph):
    nodes = list(graph.keys())
    headers = [get_initials(n) for n in nodes]
    matrix = []
    for u in nodes:
        row = []
        for v in nodes:
            cost = graph[u].get(v, "-")
            # abbreviate long names in matrix by just showing cost
            row.append(str(cost))
        matrix.append(row)
    print("\n=== Cost / Transition Matrix ===")
    print(tabulate(matrix, headers=headers, showindex=[
          n for n in nodes], tablefmt="grid"))
    print("\n Note:- '-' = No Route Available")


print_cost_matrix(graph)

# ==============================
# Goal Test
# ==============================


def goal_test(all_nodes, visited, goal_type="visit_all", current=None, goal_node=None):
    if goal_type == "visit_all":
        return visited == all_nodes
    elif goal_type == "custom_end":
        return visited == all_nodes and current == goal_node
    return False

# ==============================
# DFS Implementation
# ==============================


def dfs(graph, start, goal_type="visit_all", goal_node=None):
    all_nodes = set(graph.keys())
    start_time = time.perf_counter()
    stack = [(start, [start], {start}, 0)]
    best_path, best_cost = None, float("inf")
    nodes_expanded, max_depth = 0, 0

    while stack:
        node, path, visited, cost = stack.pop()
        nodes_expanded += 1
        max_depth = max(max_depth, len(path))

        if goal_test(all_nodes, visited, goal_type, node, goal_node):
            if cost < best_cost:
                best_cost = cost
                best_path = path
            continue

        for neighbor, weight in successors(graph, node, visited):
            stack.append(
                (neighbor, path + [neighbor], visited | {neighbor}, cost + weight))

    end_time = time.perf_counter()
    return best_path, best_cost, nodes_expanded, max_depth, (end_time - start_time)

# ==============================
# RBFS Implementation
# ==============================


def mst_cost(graph, nodes):
    if len(nodes) <= 1:
        return 0
    nodes = list(nodes)
    in_mst = {nodes[0]}
    total_cost = 0
    while len(in_mst) < len(nodes):
        min_edge = (None, math.inf)
        for u in in_mst:
            for v, w in graph[u].items():
                if v in nodes and v not in in_mst and w < min_edge[1]:
                    min_edge = (v, w)
        v, w = min_edge
        if v is None:
            break
        in_mst.add(v)
        total_cost += w
    return total_cost


def heuristic(graph, current, unvisited):
    if not unvisited:
        return 0
    return mst_cost(graph, unvisited) + min((w for v, w in graph[current].items() if v in unvisited), default=0)


def rbfs(graph, start, goal_type="visit_all", goal_node=None):
    all_nodes = set(graph.keys())
    start_time = time.perf_counter()
    nodes_expanded = 0

    def rbfs_rec(path, g, f_limit, visited):
        nonlocal nodes_expanded
        node = path[-1]
        nodes_expanded += 1

        if goal_test(all_nodes, visited, goal_type, node, goal_node):
            return path, g, None

        successors_list = []
        for neighbor, w in successors(graph, node, visited):
            new_g = g + w
            h = heuristic(graph, neighbor, all_nodes - visited - {neighbor})
            f = max(new_g + h, g + w)
            successors_list.append(
                [f, neighbor, new_g, path + [neighbor], visited | {neighbor}])

        if not successors_list:
            return None, math.inf, None
        successors_list.sort(key=lambda x: x[0])

        while True:
            best_f, _, best_g, best_path, best_visited = successors_list[0]
            if best_f > f_limit:
                return None, best_f, None

            alternative = successors_list[1][0] if len(
                successors_list) > 1 else math.inf
            result, result_cost, result_path = rbfs_rec(
                best_path, best_g, min(f_limit, alternative), best_visited)
            if result is not None:
                return result, result_cost, result_path
            successors_list[0][0] = result_cost
            successors_list.sort(key=lambda x: x[0])

    result, cost, _ = rbfs_rec([start], 0, math.inf, {start})
    end_time = time.perf_counter()
    return result, cost, nodes_expanded, len(result), (end_time - start_time)

# ==============================
# Complexity Functions
# ==============================


def compute_complexity(result, graph, algo_name):
    n = len(graph)
    total_degree = sum(len(neigh) for neigh in graph.values())
    b = total_degree / n
    d = n
    time_formula = f"O({b:.2f}^{d})"
    space_formula = f"O({d})" if algo_name == "DFS" else f"O({b:.2f}*{d})"
    # compute actual values
    time_actual = b**d
    space_actual = d if algo_name == "DFS" else b*d
    return {
        "Algorithm": algo_name,
        "Nodes (n)": n,
        "Branching Factor (b)": f"{b:.2f}",
        "Depth (d)": d,
        "Time Complexity": f"{time_formula} = {time_actual:.0f}",
        "Space Complexity": f"{space_formula} = {space_actual:.0f}",
        "Empirical Nodes Expanded": result[2],
        "Empirical Max Depth": result[3],
        "Execution Time (s)": f"{result[4]:.6f}"
    }

# ==============================
# Goal Choice
# ==============================


print("\nGoal Options:")
print("1. Visit all nodes and stop at last node")
print("2. Visit all nodes and end at custom node")

while True:
    goal_choice = input("Enter goal type (1/2): ").strip()
    if goal_choice in ("1", "2"):
        break
    else:
        print("⚠️ Invalid choice. Please enter 1 or 2.")

if goal_choice == "1":
    goal_type, goal_node = "visit_all", None

elif goal_choice == "2":
    # Ensure a valid goal node is chosen
    while True:
        print("\nChoose Goal Node:")
        for idx, node in enumerate(nodes_list, 1):
            print(f"{idx}. {node}")

        goal_idx_str = input("Enter goal node index: ").strip()
        if not goal_idx_str.isdigit():
            print("⚠️ Please enter a valid number.")
            continue

        goal_idx = int(goal_idx_str) - 1
        if 0 <= goal_idx < len(nodes_list):
            goal_node = nodes_list[goal_idx]
            if goal_node == start_node:
                print(
                    "⚠️ Goal node cannot be the same as the start node. Choose another.")
                continue
            goal_type = "custom_end"
            break
        else:
            print(
                f"⚠️ Invalid index. Enter a number between 1 and {len(nodes_list)}.")

# ==============================
# Run Algorithms
# ==============================
dfs_result = dfs(graph, start_node, goal_type, goal_node)
rbfs_result = rbfs(graph, start_node, goal_type, goal_node)

# ==============================
# Paths and Costs
# ==============================
print("\n=== Paths and Costs ===")
print(f"DFS Path: {' -> '.join(dfs_result[0])} | Cost: {dfs_result[1]}")
print(f"RBFS Path: {' -> '.join(rbfs_result[0])} | Cost: {rbfs_result[1]}")

# ==============================
# Comparative Table
# ==============================
dfs_metrics = compute_complexity(dfs_result, graph, "DFS")
rbfs_metrics = compute_complexity(rbfs_result, graph, "RBFS")

comparison_table = []
for key in dfs_metrics:
    if key not in ["Algorithm"]:  # remove Algorithm row if needed
        comparison_table.append([key, dfs_metrics[key], rbfs_metrics[key]])

print("\n=== Comparative Complexity Table ===")
print(tabulate(comparison_table, headers=[
      "Metric", "DFS", "RBFS"], tablefmt="grid"))
